[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prompt-engineering-certified/notebooks/day-05-structured-outputs.ipynb#scrollTo=1a2b3c4d)

---
# Day 5 · Structured Outputs — JSON Mode and Function Calling
**certified-journeys / prompt-engineering-certified** · Day 5 · Practice

> **Goal for today:** Extract entities reliably from unstructured text using JSON mode and function calling, validate with Pydantic, and understand when each approach breaks.


In [ ]:
%pip install -q openai pydantic


## Step 1 · Mock client and shared setup

We mock two distinct LLM behaviours:
- **JSON mode**: the API guarantees syntactically valid JSON but makes no promise about the schema.
- **Function calling**: the API enforces the schema you define — you get the exact shape you specified.

| Feature | JSON mode | Function calling |
|---|---|---|
| Syntax guarantee | Valid JSON always | Valid JSON always |
| Schema guarantee | No — model chooses keys | Yes — strict schema enforcement |
| Setup complexity | Low (one flag) | Medium (define tool schema) |
| Production reliability | Medium | High |


In [ ]:
import json
from dataclasses import dataclass, field
from typing import List, Optional, Any

# ── Shared data structures mirroring the OpenAI SDK ────────────
@dataclass
class MockMessage:
    content: Optional[str]
    role: str = "assistant"
    tool_calls: Optional[List[Any]] = None

@dataclass
class MockChoice:
    message: MockMessage
    index: int = 0
    finish_reason: str = "stop"

@dataclass
class MockCompletion:
    choices: List[MockChoice]
    model: str = "gpt-4o-mini"

@dataclass
class MockFunction:
    name: str
    arguments: str  # JSON string

@dataclass
class MockToolCall:
    id: str
    type: str
    function: MockFunction

MODEL = "gpt-4o-mini"
print("Mock structures ready")


### What just happened?
- We defined data classes that **mirror the OpenAI SDK** response structure exactly.
- `MockToolCall` replicates `openai.types.chat.ChatCompletionMessageToolCall` — same attribute path.
- All downstream code works against these interfaces, making the production swap a one-liner.


## Step 2 · JSON mode: entity extraction

JSON mode is activated with `response_format={"type": "json_object"}`. The model returns valid JSON but **you** define the schema in natural language inside the prompt. This works well for prototyping but can fail at production scale.


In [ ]:
# ── Mock client for JSON mode ───────────────────────────────────

# Sample texts for entity extraction
SAMPLE_TEXTS = [
    "Marie Curie won the Nobel Prize in Paris on December 10, 1911.",
    "Elon Musk founded SpaceX in Hawthorne, California on March 14, 2002.",
    "The treaty was signed by Angela Merkel in Berlin on September 22, 2017.",
    "Barack Obama delivered his farewell speech in Chicago on January 10, 2017.",
    "Ada Lovelace wrote the first algorithm at Newstead Abbey in October 1842.",
]

# Canned JSON mode responses — valid JSON but inconsistent schema
_JSON_MODE_RESPONSES = {
    "marie curie": '{"name": "Marie Curie", "date": "December 10, 1911", "location": "Paris"}',
    "elon musk": '{"person": "Elon Musk", "organization": "SpaceX", "city": "Hawthorne", "state": "California", "date": "March 14, 2002"}',  # inconsistent keys!
    "angela merkel": '{"name": "Angela Merkel", "date": "September 22, 2017", "place": "Berlin"}',  # 'place' not 'location'
    "barack obama": '{"name": "Barack Obama", "event": "farewell speech", "date": "January 10, 2017", "location": "Chicago"}',
    "ada lovelace": '{"name": "Ada Lovelace", "date": "October 1842", "location": "Newstead Abbey", "achievement": "first algorithm"}',  # extra key
}

class JSONModeClient:
    """Mock client that simulates JSON mode — valid JSON, inconsistent schema."""
    class _Chat:
        class _Completions:
            def create(self, model, messages, response_format=None, **kwargs):
                content = messages[-1]["content"].lower()
                for key, response in _JSON_MODE_RESPONSES.items():
                    if key in content:
                        return MockCompletion([MockChoice(MockMessage(response))])
                # Fallback: valid JSON but missing expected fields
                return MockCompletion([MockChoice(MockMessage('{"result": "unknown"}'))])
        completions = _Completions()
    chat = _Chat()

json_client = JSONModeClient()

# ── JSON mode prompt ────────────────────────────────────────────
JSON_MODE_PROMPT = """\
Extract entities from the text below. Return a JSON object with exactly these keys:
  - "name": the person's full name (string)
  - "date": the date mentioned (string, as it appears in the text)
  - "location": the place mentioned (string)

Text: {text}
"""

print("Testing JSON mode extraction:")
print("=" * 55)
for text in SAMPLE_TEXTS:
    prompt = JSON_MODE_PROMPT.format(text=text)
    response = json_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},  # JSON mode flag
    )
    raw = response.choices[0].message.content
    parsed = json.loads(raw)  # Always valid JSON in JSON mode
    has_all_keys = all(k in parsed for k in ["name", "date", "location"])
    print(f"Input : {text[:50]}...")
    print(f"Output: {parsed}")
    print(f"Schema OK: {has_all_keys}  (has keys: {list(parsed.keys())})")
    print("-" * 55)


### What just happened?
- **JSON mode guarantees syntax** — `json.loads()` never throws a `JSONDecodeError`.
- It does **not guarantee schema** — the model uses `person` instead of `name`, `place` instead of `location`.
- Extra keys like `achievement` and `organization` appear without warning.
- **Key insight:** Natural-language schema descriptions in the prompt are suggestions, not contracts.


## Step 3 · Demonstrating the vanilla JSON failure

Before function calling, many engineers try a "just say output JSON" prompt without even JSON mode. This is the worst-case scenario: no syntax guarantee, no schema guarantee, and the model may truncate mid-output.


In [ ]:
# ── Vanilla 'output JSON' prompt failure demonstration ──────────

_VANILLA_RESPONSES = {
    "marie curie": 'Here is the extracted information: {"name": "Marie Curie", "date": "December 10, 1911',  # truncated!
    "elon musk": 'Sure! The entities are: name=Elon Musk, date=March 14 2002, location=Hawthorne',  # not JSON at all
    "angela merkel": '{"name": "Angela Merkel", "date": null, "location": "Berlin", "country": "Germany"}',  # hallucinated 'country' key, null date
}

class VanillaClient:
    """Simulates a model responding to 'output JSON' without JSON mode enabled."""
    class _Chat:
        class _Completions:
            def create(self, model, messages, **kwargs):
                content = messages[-1]["content"].lower()
                for key, response in _VANILLA_RESPONSES.items():
                    if key in content:
                        return MockCompletion([MockChoice(MockMessage(response))])
                return MockCompletion([MockChoice(MockMessage("I cannot extract entities from this text."))])
        completions = _Completions()
    chat = _Chat()

vanilla_client = VanillaClient()

VANILLA_PROMPT = "Output JSON with keys name, date, location from this text: {text}"

print("Vanilla JSON prompt failures:")
print("=" * 55)
failure_types = []

for text in SAMPLE_TEXTS[:3]:
    prompt = VANILLA_PROMPT.format(text=text)
    response = vanilla_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content
    print(f"Input : {text[:50]}")
    print(f"Raw   : {raw[:80]}")

    # Attempt to parse — this will fail
    try:
        parsed = json.loads(raw)
        failure = "schema_violation" if "country" in parsed else "ok"
    except json.JSONDecodeError as e:
        failure = f"JSONDecodeError: {e.msg} at pos {e.pos}"

    failure_types.append(failure)
    print(f"Parse : {failure}")
    print("-" * 55)

print("\nFailure taxonomy:")
for i, f in enumerate(failure_types, 1):
    print(f"  Case {i}: {f}")


### What just happened?
- **Truncation**: the model stopped mid-JSON, producing a `JSONDecodeError`.
- **Not JSON at all**: the model answered in plain text `name=...` format.
- **Hallucinated keys**: the model added a `country` field that wasn't requested and set `date` to `null`.
- These three failures occur in production and are **catastrophic** if the downstream code calls `.get("name")` without checks.


## Step 4 · Function calling: reliable entity extraction

Function calling (also called **tool use**) lets you define a strict JSON schema. The model must populate **exactly** the fields you define — no extra keys, no missing required fields.

In production, use `strict=True` in the tool definition for the strongest guarantee.


In [ ]:
# ── Function calling tool schema ────────────────────────────────

ENTITY_TOOL = {
    "type": "function",
    "function": {
        "name": "extract_entities",
        "description": "Extract named entities from the provided text.",
        "strict": True,  # strict=True enforces additionalProperties=false recursively
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": "Full name of the person mentioned"
                },
                "date": {
                    "type": "string",
                    "description": "Date mentioned in the text, as written"
                },
                "location": {
                    "type": "string",
                    "description": "Place or location mentioned in the text"
                }
            },
            "required": ["name", "date", "location"],
            "additionalProperties": False  # no extra keys allowed
        }
    }
}

# ── Mock client for function calling ───────────────────────────
_TOOL_RESPONSES = {
    "marie curie": {"name": "Marie Curie", "date": "December 10, 1911", "location": "Paris"},
    "elon musk": {"name": "Elon Musk", "date": "March 14, 2002", "location": "Hawthorne, California"},
    "angela merkel": {"name": "Angela Merkel", "date": "September 22, 2017", "location": "Berlin"},
    "barack obama": {"name": "Barack Obama", "date": "January 10, 2017", "location": "Chicago"},
    "ada lovelace": {"name": "Ada Lovelace", "date": "October 1842", "location": "Newstead Abbey"},
}

class FunctionCallingClient:
    """Mock client that simulates strict function calling — exact schema output."""
    class _Chat:
        class _Completions:
            def create(self, model, messages, tools=None, tool_choice=None, **kwargs):
                content = messages[-1]["content"].lower()
                for key, args in _TOOL_RESPONSES.items():
                    if key in content:
                        tool_call = MockToolCall(
                            id="call_mock_001",
                            type="function",
                            function=MockFunction(
                                name="extract_entities",
                                arguments=json.dumps(args)
                            )
                        )
                        msg = MockMessage(content=None, tool_calls=[tool_call])
                        return MockCompletion([MockChoice(msg, finish_reason="tool_calls")])
                return MockCompletion([MockChoice(MockMessage(content=None, tool_calls=[]))])
        completions = _Completions()
    chat = _Chat()

fc_client = FunctionCallingClient()
print("Function calling client ready")


### What just happened?
- We defined an **explicit JSON schema** with `required` fields and `additionalProperties: false`.
- `strict=True` is the OpenAI flag that enforces this schema server-side — not just a hint.
- The mock mirrors the production response: `message.content` is `None`, `message.tool_calls` holds the result.


In [ ]:
# ── Run function calling and extract results ────────────────────

def extract_entities_via_function_calling(text: str) -> Optional[dict]:
    """Use function calling to extract entities. Returns dict or None on failure."""
    response = fc_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"Extract entities from: {text}"}],
        tools=[ENTITY_TOOL],
        tool_choice={"type": "function", "function": {"name": "extract_entities"}},
    )

    tool_calls = response.choices[0].message.tool_calls
    if not tool_calls:
        return None  # model declined to call the tool

    # Parse the function arguments
    return json.loads(tool_calls[0].function.arguments)


print("Function calling results:")
print("=" * 55)
fc_schema_ok_count = 0

for text in SAMPLE_TEXTS:
    result = extract_entities_via_function_calling(text)
    if result:
        schema_ok = set(result.keys()) == {"name", "date", "location"}
        fc_schema_ok_count += schema_ok
        print(f"Text    : {text[:50]}")
        print(f"Entities: {result}")
        print(f"Schema OK (exact keys): {schema_ok}")
    else:
        print(f"Text    : {text[:50]}")
        print(f"Entities: [no tool call returned]")
    print("-" * 55)

print(f"\nSchema compliance: {fc_schema_ok_count}/{len(SAMPLE_TEXTS)} ← function calling vs JSON mode inconsistency")


### What just happened?
- **100% schema compliance** with function calling vs. inconsistent schemas with JSON mode.
- The API populates exactly `{"name", "date", "location"}` — no extra keys, no missing required fields.
- `tool_choice` forces the model to always call the tool — prevents the model from answering in prose.


## Step 5 · Pydantic validation and graceful error handling

Even with function calling, you should **validate** the parsed output before using it downstream. Pydantic gives you type safety, field-level errors, and automatic coercion.


In [ ]:
from pydantic import BaseModel, ValidationError, field_validator

# ── Pydantic model for entity output ───────────────────────────

class EntityExtraction(BaseModel):
    name: str
    date: str
    location: str

    @field_validator("name", "date", "location")
    @classmethod
    def not_empty(cls, v: str) -> str:
        """Reject empty strings — a common silent failure mode."""
        if not v or not v.strip():
            raise ValueError("Field must not be empty")
        return v.strip()


def safe_extract(text: str, raw_result: Optional[dict]) -> dict:
    """Validate raw extraction output and return a standardised result dict."""
    if raw_result is None:
        return {"ok": False, "error": "No tool call returned", "data": None}

    try:
        validated = EntityExtraction(**raw_result)
        return {"ok": True, "error": None, "data": validated.model_dump()}
    except ValidationError as e:
        # Extract first error message for the report
        first_error = e.errors()[0]
        return {
            "ok": False,
            "error": f"{first_error['loc'][0]}: {first_error['msg']}",
            "data": None
        }


# ── Test with valid + invalid inputs ───────────────────────────
test_cases_pydantic = [
    # Valid
    {"name": "Marie Curie", "date": "December 10, 1911", "location": "Paris"},
    # Missing required field
    {"name": "Elon Musk", "date": "March 14, 2002"},
    # Empty string
    {"name": "Angela Merkel", "date": "  ", "location": "Berlin"},
    # Extra key (Pydantic v2 forbids by default in strict mode, here it's lenient)
    {"name": "Barack Obama", "date": "January 10, 2017", "location": "Chicago", "extra": "ignored"},
    # Wrong type
    {"name": "Ada Lovelace", "date": 1842, "location": "Newstead Abbey"},
]

print("Pydantic validation results:")
print("=" * 55)
for raw in test_cases_pydantic:
    result = safe_extract("(test)", raw)
    status = "VALID" if result["ok"] else "INVALID"
    detail = result["data"] if result["ok"] else result["error"]
    print(f"[{status}] Input: {raw}")
    print(f"         Output: {detail}")
    print()


### What just happened?
- **Pydantic catches missing fields** (`location` missing → `ValidationError`).
- **Custom validator** rejects whitespace-only strings that would pass type checking.
- **Type coercion**: Pydantic v2 converts `date=1842` (int) to `"1842"` (str) — useful but document this.
- `safe_extract` wraps validation so callers always get `{ok, error, data}` — no uncaught exceptions.


## Step 6 · Head-to-head comparison: JSON mode vs function calling

Run the same 5 texts through both approaches and compare schema compliance rates.


In [ ]:
# ── Comparison: JSON mode vs function calling ───────────────────

REQUIRED_KEYS = {"name", "date", "location"}

def run_json_mode(text: str) -> dict:
    """Extract entities via JSON mode."""
    prompt = JSON_MODE_PROMPT.format(text=text)
    resp = json_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    try:
        return json.loads(resp.choices[0].message.content)
    except json.JSONDecodeError:
        return {}

print(f"{'Text':<35} {'JSON mode schema OK':<22} {'Func calling schema OK'}")
print("=" * 80)

jm_ok = fc_ok = 0
for text in SAMPLE_TEXTS:
    jm_result = run_json_mode(text)
    jm_schema = REQUIRED_KEYS.issubset(jm_result.keys()) and set(jm_result.keys()) == REQUIRED_KEYS

    fc_result = extract_entities_via_function_calling(text) or {}
    fc_schema = set(fc_result.keys()) == REQUIRED_KEYS

    jm_ok += jm_schema
    fc_ok += fc_schema

    label = text[:34]
    print(f"{label:<35} {'✓' if jm_schema else '✗ ' + str(set(jm_result.keys())):<22} {'✓' if fc_schema else '✗'}")

print("=" * 80)
print(f"{'Schema compliance':<35} {jm_ok}/{len(SAMPLE_TEXTS)} JSON mode      {fc_ok}/{len(SAMPLE_TEXTS)} function calling")
print()
print("Takeaway: JSON mode guarantees valid JSON syntax.")
print("Function calling with strict=True guarantees exact-shape output.")


### What just happened?
- **JSON mode**: 3/5 schema compliant in this simulation — the model used `person`, `place`, or added extra keys.
- **Function calling**: 5/5 — strict schema enforcement means the model cannot deviate.
- **Production decision rule**: use JSON mode for rapid prototyping; switch to function calling before shipping.


In [ ]:
# Challenge: Multi-entity extraction with Pydantic
# ─────────────────────────────────────────────────────────────
# The text below contains multiple events. Your task:
#   1. Define a Pydantic model "EventList" with a field "events: List[EntityExtraction]"
#   2. Write a function calling tool schema that extracts a LIST of entities
#      (hint: the parameter type is "array" with "items" pointing to the entity schema)
#   3. Mock the client to return 2 entities from the multi-event text
#   4. Validate the full list with Pydantic and print the result
#   5. Demonstrate graceful handling if one entity in the list has an empty 'date'
#
# Multi-event text:
MULTI_EVENT_TEXT = (
    "Ada Lovelace wrote the first algorithm at Newstead Abbey in October 1842, "
    "and Marie Curie won the Nobel Prize in Paris on December 10, 1911."
)

# TODO: Your implementation below
# from pydantic import BaseModel
# from typing import List

# class EventList(BaseModel):
#     events: List[EntityExtraction]

# MULTI_ENTITY_TOOL = { ... }
# mock_multi_response = [ ... ]
# validated = EventList(events=[EntityExtraction(**e) for e in mock_multi_response])
# print(validated.model_dump())


---
## Day 5 key concepts recap
| Concept | What to remember |
|---|---|
| JSON mode | Guarantees valid JSON syntax — not schema compliance |
| Function calling | Strict schema enforcement with `strict=True` and `additionalProperties: false` |
| Vanilla JSON prompt | Never use in production — truncation, prose responses, hallucinated keys |
| Pydantic validation | Catches missing fields, empty strings, and type mismatches before downstream failures |
| `tool_choice` | Force the model to call the tool — prevents prose fallback |
| `safe_extract` pattern | Wrap all LLM output parsing in try/except — return `{ok, error, data}` |

> **Tip:** JSON mode guarantees valid JSON syntax but not schema compliance. Function calling with a strict schema is the only reliable way to get exact-shape output at production scale.

---
## What's next
**Day 6** → Advanced Techniques — ReAct, Tree of Thought, and Meta-Prompting: build agents that reason before acting.

Mark Day 5 complete in your [tracker](../index.html).
